In [194]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
from io import StringIO
from hallmark.eht_datatree import build_tree, build_repo
from hallmark import Repo
from collections import defaultdict

pd.set_option("display.width", 300)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

DATASETS = ["EHTC_FirstSgrAResults_May2022","EHTC_CenA2017_Jul2021",
           "EHTC_First3C279Results_May2020","EHTC_FirstM87Results_Apr2019",
           "EHTC_FirstSgrAPol_Mar2024","EHTC_M87-2018_Mar2024",
           "EHTC_M87mwl2017_Apr2021","EHTC_M87mwl2018_Dec2024",
           "EHTC_M87pol2017_Nov2023","EHTC_metadata2018_Dec2023",
           "EHTC_MonitoringM87_Sep2020","EHTC_SgrAmwl2017_May2022"]
idx = 0
FMT = "ER6_SGRA_2017_{scan}_{band}_{pipeline}_netcal-LMTcal-{method}_StokesI"
ROOT = Path(f"~/{DATASETS[idx]}").expanduser()
REPO_PATH = Path(f"~/{DATASETS[idx]}_hallmark").expanduser()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


A data tree will be built for a branch for each different fmt, and a sub-branch for each stem of that fmt

In [195]:
tree = build_tree(ROOT, fmt=FMT, data_type="L2")

print("Tree structure:")

print("\n  meta:")
print()
for _, row in tree["meta"].iterrows():
    print(f"    {row['path']}")

for fmt_str, fmt_dict in tree["data"].items():
    print(f"\n  fmt: {fmt_str}")
    print()
    for stem_key, stem_pf in fmt_dict.items():
        print(f"    {stem_key:<35} {len(stem_pf)} file(s)")

Tree structure:

  meta:

    EHTC_FirstSgrAResults_May2022_csv/EHTC_FirstSgrAResults_May2022/csv/dump_csv.py
    EHTC_FirstSgrAResults_May2022_csv/EHTC_FirstSgrAResults_May2022/csv_besttime/dump_csv.py
    EHTC_FirstSgrAResults_May2022_csv/EHTC_FirstSgrAResults_May2022/csv_norm/dump_csv.py
    EHTC_FirstSgrAResults_May2022_txt/EHTC_FirstSgrAResults_May2022/txt/dump_txt.py
    EHTC_FirstSgrAResults_May2022_txt/EHTC_FirstSgrAResults_May2022/txt_besttime/dump_txt.py
    EHTC_FirstSgrAResults_May2022_txt/EHTC_FirstSgrAResults_May2022/txt_norm/dump_txt.py
    EHTC_FirstSgrAResults_May2022_uvfits/EHTC_FirstSgrAResults_May2022/uvfits/convert_stokesI.py
    EHTC_FirstSgrAResults_May2022_uvfits/EHTC_FirstSgrAResults_May2022/uvfits_besttime/convert_stokesI.py
    EHTC_FirstSgrAResults_May2022_uvfits/EHTC_FirstSgrAResults_May2022/uvfits_norm/convert_stokesI.py
    INVENTORY.txt
    LICENSE.txt
    README.md
    run.sh

  fmt: ER6_SGRA_2017_{scan}_{band}_{pipeline}_netcal-LMTcal-{method}_StokesI


if no fmt is entered, they can still be detected with generic parameters

In [196]:
tree_no_fmt = build_tree(ROOT, fmt=None, data_type="L2")

print("\nTree structure:")
for fmt_str, fmt_dict in tree_no_fmt["data"].items():
    print(f"\n fmt: {fmt_str}")
    print()
    for stem_key, stem_pf in fmt_dict.items():
        print(f"    {stem_key:<35} {len(stem_pf)} file(s)")


Tree structure:

 fmt: ER6_SGRA_2017_{p0}_{p1}_{p2}_netcal-LMTcal-{p3}_StokesI

    097_hi_hops_besttime                3 file(s)
    097_hi_casa_besttime                3 file(s)
    097_lo_casa_besttime                3 file(s)
    097_lo_hops_besttime                3 file(s)
    096_lo_casa_norm                    3 file(s)
    097_lo_casa_norm                    3 file(s)
    096_hi_hops_norm                    3 file(s)
    097_lo_hops_norm                    3 file(s)
    097_hi_hops_norm                    3 file(s)
    097_hi_casa_norm                    3 file(s)
    096_lo_hops_norm                    3 file(s)
    096_hi_casa_norm                    3 file(s)
    096_hi_hops                         3 file(s)
    097_lo_hops                         3 file(s)
    096_lo_casa                         3 file(s)
    097_hi_hops                         3 file(s)
    096_lo_hops                         3 file(s)
    096_hi_casa                         3 file(s)
    097_lo_casa    

This datatree can then be transformed into a repo using the existing branches and stems. Next I will try to streamline this into one process staight into the repo without building the in-memory tree

In [197]:
repo = build_repo(
    tree=tree,
    repo_path=REPO_PATH,
    dataset_name=DATASETS[idx],
    worktree_root=ROOT,
    overwrite=True,
)
print("Repo built at:", repo.dothm.path)

branches = repo.branches()
print(f"Current branch: {branches['current']}")
print(f"\nTotal branches: {len(branches['names'])}")

print("\nmain/")
print("  meta.yaml:")
for f in repo.state.meta.get("files", []):
    print(f"    {Path(f).name}")

fmt_prefixes = sorted({
    name.split("/")[0]
    for name in branches["names"]
    if name != "main"
})

for fmt in fmt_prefixes:
    first_stem = next(
        name for name in branches["names"]
        if name.startswith(fmt + "/")
    )
    repo.dothm.git.checkout(first_stem)
    repo.state = repo.dothm.load()
    real_fmt = repo.state.config["data"][0]["fmt"]
    stem_count = sum(1 for n in branches["names"] if n.startswith(fmt + "/"))
    print(f"\n{real_fmt}/  \n  ({stem_count} stems)")

repo.dothm.git.checkout("main")
repo.state = repo.dothm.load()



Repo built at: /home/samj98/EHTC_FirstSgrAResults_May2022_hallmark/.hm
Current branch: main

Total branches: 21

main/
  meta.yaml:
    dump_csv.py
    dump_csv.py
    dump_csv.py
    dump_txt.py
    dump_txt.py
    dump_txt.py
    convert_stokesI.py
    convert_stokesI.py
    convert_stokesI.py
    INVENTORY.txt
    LICENSE.txt
    README.md
    run.sh

ER6_SGRA_2017_{scan}_{band}_{pipeline}_netcal-LMTcal-{method}_StokesI/  
  (20 stems)


In [198]:
# get unique fmt prefixes
fmt_prefixes = sorted({
    name.split("/")[0]
    for name in branches["names"]
    if name != "main"
})

print()
for fmt_prefix in fmt_prefixes:
    # get real fmt string from first stem's config
    first_stem = next(
        b for b in sorted(branches["names"])
        if b.startswith(fmt_prefix + "/")
    )
    real_fmt = repo.dothm.git.show(f"{first_stem}:config.yml")
    import yaml
    config = yaml.safe_load(real_fmt)
    fmt_str = config["data"][0]["fmt"]
    
    # get stems for this fmt
    fmt_branches = sorted(b for b in branches["names"] 
                          if b.startswith(fmt_prefix + "/"))
    
    # get param cols from first stem
    first_data = pd.read_csv(
        StringIO(repo.dothm.git.show(f"{first_stem}:data.tsv")),
        sep="\t", dtype=str
    )
    param_cols = [c for c in first_data.columns if c != "sha1"]
    
    header = f"  {'STEM':<35} {'FILES':<8} " + "  ".join(f"{col.upper():<10}" 
                                                         for col in param_cols)
    print(f"fmt: {fmt_str}")
    print(header)
    print("  " + "=" * (len(header) - 2))
    
    for branch in fmt_branches:
        stem = branch.split("/")[-1]
        data = pd.read_csv(
            StringIO(repo.dothm.git.show(f"{branch}:data.tsv")),
            sep="\t", dtype=str
        )
        row = data.iloc[0]
        params = "  ".join(f"{str(row.get(col, 'NaN')):<10}" for col in param_cols)
        print(f"  {stem:<35} {len(data):<8} {params}")
    
    print(f"  Total: {len(fmt_branches)} stems\n")


fmt: ER6_SGRA_2017_{scan}_{band}_{pipeline}_netcal-LMTcal-{method}_StokesI
  STEM                                FILES    SCAN        BAND        PIPELINE    METHOD    
  096_hi_casa                         3        096         hi          casa        nan       
  096_hi_casa_norm                    3        096         hi          casa        norm      
  096_hi_hops                         3        096         hi          hops        nan       
  096_hi_hops_norm                    3        096         hi          hops        norm      
  096_lo_casa                         3        096         lo          casa        nan       
  096_lo_casa_norm                    3        096         lo          casa        norm      
  096_lo_hops                         3        096         lo          hops        nan       
  096_lo_hops_norm                    3        096         lo          hops        norm      
  097_hi_casa                         3        097         hi          casa   

In [199]:
for b in sorted(b for b in branches["names"] if "m87_2017" in b):
    print(repr(b))

In [200]:
target_branch = next(b for b in sorted(branches["names"]) if b != "main")

repo.dothm.git.checkout(target_branch)
repo.state = repo.dothm.load()

print(f"Branch: {target_branch.split('/')[-1]}")
print(f"\nConfig:")
print(f"  fmt: {repo.state.config['data'][0]['fmt']}")
print(f"  encoding: {repo.state.config['data'][0]['encoding']}")
print(f"  remote: {repo.state.config.get('remote')}")
print(f"\nData:")
print(repo.state.data)

# return to main
repo.dothm.git.checkout("main")
repo.state = repo.dothm.load()

Branch: 096_hi_casa

Config:
  fmt: ER6_SGRA_2017_{scan}_{band}_{pipeline}_netcal-LMTcal-{method}_StokesI
  encoding: None
  remote: None

Data:
                                       sha1 scan band pipeline method
0  d291317a6e342a196b286078fc424d938b570050  096   hi     casa    NaN
1  8276698b2f2d702b6416fc5d9678a2d829d25f09  096   hi     casa    NaN
2  45d3a32340b519357ff97e19689fc59b4a56703b  096   hi     casa    NaN
